# 🔍 Agente RAG con LangChain + Mistral

## Arquitectura completa
```
CSV
 │
 ├─ 1. DOCUMENTOS   → convierte filas del CSV en documentos de texto
 ├─ 2. EMBEDDINGS   → MistralAIEmbeddings convierte cada doc en vector
 ├─ 3. VECTOR STORE → FAISS guarda y busca por similitud
 ├─ 4. RETRIEVER    → busca los docs más relevantes para cada pregunta
 ├─ 5. PROMPT       → plantilla que combina pregunta + contexto
 ├─ 6. LLM          → Mistral genera la respuesta final
 └─ 7. RAG CHAIN    → une todo con LCEL usando el operador |  
                      rag_chain.invoke({"question": "..."})
```
---

## CELDA 1 — Instalación
> Reinicia el runtime cuando termine.

In [1]:
!pip install -q langchain langchain-mistralai langchain-community faiss-cpu
print('✅ Listo. Reinicia: Entorno de ejecución → Reiniciar sesión')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
✅ Listo. Reinicia: Entorno de ejecución → Reiniciar sesión


## CELDA 2 — Imports

In [2]:
import pandas as pd
import numpy as np
from google.colab import files, userdata

# LangChain — LLM y Embeddings
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings

# LangChain — Vector Store
from langchain_community.vectorstores import FAISS

# LangChain — Documentos
from langchain_core.documents import Document

# LangChain — Prompt (movido a langchain_core)
from langchain_core.prompts import ChatPromptTemplate

# LangChain — Cadena RAG moderna (LCEL)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print('✅ Imports correctos')

✅ Imports correctos


## CELDA 3 — Cargar el CSV

In [3]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename, encoding='latin-1')
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

print(f'✅ Dataset cargado: {filename}')
print(f'   {df.shape[0]:,} filas × {df.shape[1]} columnas')
df.head(3)

Saving sales_data_sample.csv to sales_data_sample.csv
✅ Dataset cargado: sales_data_sample.csv
   2,823 filas × 25 columnas


,ordernumber,quantityordered,priceeach,orderlinenumber,sales,orderdate,status,qtr_id,month_id,year_id,...,addressline1,addressline2,city,state,postalcode,country,territory,contactlastname,contactfirstname,dealsize
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium


## CELDA 4 — Convertir el CSV en Documentos

RAG no trabaja con DataFrames — trabaja con **documentos de texto**.
Cada documento es un fragmento temático del dataset con sus metadatos.

```
DataFrame ──→ Chunks temáticos ──→ Lista de Document(page_content, metadata)
```

In [4]:
def crear_documentos(df: pd.DataFrame) -> list:
    """
    Convierte el DataFrame en una lista de Documents de LangChain.
    Cada Document tiene:
      - page_content : texto que describe un aspecto del dataset
      - metadata     : información sobre el tema del documento
    """
    docs = []

    # Doc 1: KPIs generales
    if 'sales' in df.columns:
        docs.append(Document(
            page_content=(
                f"Resumen general de ventas del dataset. "
                f"Total de órdenes: {len(df):,}. "
                f"Venta total acumulada: ${df['sales'].sum():,.2f}. "
                f"Venta promedio por orden: ${df['sales'].mean():,.2f}. "
                f"Venta máxima registrada: ${df['sales'].max():,.2f}. "
                f"Venta mínima registrada: ${df['sales'].min():,.2f}."
            ),
            metadata={'tema': 'KPIs generales', 'fuente': 'dataset'}
        ))

    # Doc 2: Ventas por línea de producto
    if 'productline' in df.columns and 'sales' in df.columns:
        resumen = df.groupby('productline')['sales'].agg(['sum', 'mean', 'count'])
        texto = "Ventas desglosadas por línea de producto. "
        for prod, row in resumen.sort_values('sum', ascending=False).iterrows():
            texto += (f"{prod}: total ${row['sum']:,.2f}, "
                     f"promedio ${row['mean']:,.2f}, "
                     f"{int(row['count'])} órdenes. ")
        docs.append(Document(page_content=texto, metadata={'tema': 'productos'}))

    # Doc 3: Ventas por año
    if 'year_id' in df.columns and 'sales' in df.columns:
        resumen = df.groupby('year_id')['sales'].agg(['sum', 'count'])
        texto = "Ventas anuales históricas. "
        for yr, row in resumen.iterrows():
            texto += f"Año {int(yr)}: ${row['sum']:,.2f} en {int(row['count'])} órdenes. "
        docs.append(Document(page_content=texto, metadata={'tema': 'ventas anuales'}))

    # Doc 4: Ventas por trimestre
    if 'qtr_id' in df.columns and 'year_id' in df.columns and 'sales' in df.columns:
        resumen = df.groupby(['year_id', 'qtr_id'])['sales'].sum()
        texto = "Ventas por trimestre. "
        for (yr, q), val in resumen.items():
            texto += f"Q{int(q)} {int(yr)}: ${val:,.2f}. "
        docs.append(Document(page_content=texto, metadata={'tema': 'trimestres'}))

    # Doc 5: Top clientes
    if 'customername' in df.columns and 'sales' in df.columns:
        top = df.groupby('customername')['sales'].sum().nlargest(10)
        texto = "Los 10 clientes con mayor volumen de compras. "
        for i, (cliente, val) in enumerate(top.items(), 1):
            texto += f"{i}. {cliente}: ${val:,.2f}. "
        docs.append(Document(page_content=texto, metadata={'tema': 'clientes'}))

    # Doc 6: Ventas por país
    if 'country' in df.columns and 'sales' in df.columns:
        resumen = df.groupby('country')['sales'].agg(['sum', 'count']).sort_values('sum', ascending=False)
        texto = "Ventas por país de origen del cliente. "
        for pais, row in resumen.iterrows():
            texto += f"{pais}: ${row['sum']:,.2f} ({int(row['count'])} órdenes). "
        docs.append(Document(page_content=texto, metadata={'tema': 'países'}))

    # Doc 7: Estado de órdenes
    if 'status' in df.columns:
        resumen = df['status'].value_counts()
        texto = "Estado actual de todas las órdenes del dataset. "
        for estado, cnt in resumen.items():
            pct = cnt / len(df) * 100
            texto += f"{estado}: {cnt} órdenes ({pct:.1f}%). "
        docs.append(Document(page_content=texto, metadata={'tema': 'estado órdenes'}))

    # Doc 8: Tamaño de deals
    if 'dealsize' in df.columns and 'sales' in df.columns:
        resumen = df.groupby('dealsize')['sales'].agg(['sum', 'mean', 'count'])
        texto = "Análisis por tamaño de negocio (deal size). "
        for size, row in resumen.iterrows():
            texto += (f"{size}: ${row['sum']:,.2f} total, "
                     f"${row['mean']:,.2f} promedio, "
                     f"{int(row['count'])} órdenes. ")
        docs.append(Document(page_content=texto, metadata={'tema': 'deal size'}))

    # Doc 9: Top ciudades
    if 'city' in df.columns and 'sales' in df.columns:
        top = df.groupby('city')['sales'].sum().nlargest(10)
        texto = "Top 10 ciudades con más ventas. "
        for ciudad, val in top.items():
            texto += f"{ciudad}: ${val:,.2f}. "
        docs.append(Document(page_content=texto, metadata={'tema': 'ciudades'}))

    # Doc 10: Productos más pedidos
    if 'productcode' in df.columns and 'quantityordered' in df.columns:
        top = df.groupby('productcode')['quantityordered'].sum().nlargest(10)
        texto = "Productos con mayor cantidad de unidades ordenadas. "
        for prod, qty in top.items():
            texto += f"{prod}: {int(qty):,} unidades. "
        docs.append(Document(page_content=texto, metadata={'tema': 'productos más pedidos'}))

    return docs


documentos = crear_documentos(df)

print(f'✅ {len(documentos)} documentos creados:')
for i, doc in enumerate(documentos, 1):
    print(f'  {i}. [{doc.metadata["tema"]}] — {len(doc.page_content)} caracteres')

✅ 10 documentos creados:
  1. [KPIs generales] — 210 caracteres
  2. [productos] — 495 caracteres
  3. [ventas anuales] — 149 caracteres
  4. [trimestres] — 250 caracteres
  5. [clientes] — 439 caracteres
  6. [países] — 708 caracteres
  7. [estado órdenes] — 226 caracteres
  8. [deal size] — 230 caracteres
  9. [ciudades] — 266 caracteres
  10. [productos más pedidos] — 305 caracteres


## CELDA 5 — Embeddings + Vector Store (FAISS)

```
MistralAIEmbeddings           FAISS
  convierte texto        guarda los vectores
  en vectores numéricos  y permite buscar por
  de 1024 dimensiones    similitud coseno
```

In [6]:
MISTRAL_API_KEY = userdata.get('MISTRAL_KEY')

# ── Embeddings ────────────────────────────────────────────────────
embeddings = MistralAIEmbeddings(
    model='mistral-embed',
    api_key=MISTRAL_API_KEY
)

print('Generando embeddings y construyendo vector store...')

# ── Vector Store FAISS ────────────────────────────────────────────
# from_documents: toma los docs, genera embeddings y los indexa en FAISS
vector_store = FAISS.from_documents(
    documents=documentos,
    embedding=embeddings
)

print(f'✅ Vector store creado con {len(documentos)} vectores')
print(f'   Modelo de embeddings: mistral-embed')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

Generando embeddings y construyendo vector store...
✅ Vector store creado con 10 vectores
   Modelo de embeddings: mistral-embed


## CELDA 6 — Retriever

El **retriever** es la interfaz de búsqueda sobre el vector store.
Recibe una pregunta y devuelve los documentos más relevantes.

```
pregunta ──→ retriever.invoke(pregunta) ──→ [doc1, doc2, doc3]
```

In [7]:
# k=3: recupera los 3 documentos más similares a la pregunta
retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

# Prueba del retriever
print('═══ Prueba del retriever ═══════════════════════')
docs_recuperados = retriever.invoke('¿cuánto se vendió por año?')
for doc in docs_recuperados:
    print(f'  ✅ [{doc.metadata["tema"]}]')
print(f'\n✅ Retriever listo (k=3)')

═══ Prueba del retriever ═══════════════════════
  ✅ [ventas anuales]
  ✅ [productos]
  ✅ [KPIs generales]

✅ Retriever listo (k=3)


## CELDA 7 — LLM + Prompt

El **prompt** es la plantilla que combina la pregunta con el contexto recuperado
antes de enviarlo al LLM.

```
prompt = "Contexto: {context}\n\nPregunta: {question}"
              │                        │
         docs del retriever      tu pregunta
```

In [8]:
# ── LLM ──────────────────────────────────────────────────────────
llm = ChatMistralAI(
    model='mistral-large-latest',
    api_key=MISTRAL_API_KEY,
    temperature=0
)

# ── Prompt ────────────────────────────────────────────────────────
TEMPLATE = """Eres un analista de datos de ventas experto.
Responde siempre en español, de forma clara y concisa.
Usa $ y formato de miles para cifras.
Basa tu respuesta ÚNICAMENTE en el contexto proporcionado.
Si la información no está en el contexto, dilo claramente.

Contexto recuperado:
{context}

Pregunta: {question}

Respuesta:"""

prompt = ChatPromptTemplate.from_template(TEMPLATE)

print('✅ LLM y prompt configurados')

✅ LLM y prompt configurados


## CELDA 8 — Función para formatear documentos + Cadena RAG moderna

La cadena RAG usa **LCEL** (LangChain Expression Language) con el operador `|` para conectar los pasos:

```python
rag_chain = (
    {"context": retriever | formatear_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
```

```
pregunta
   │
   ├─→ retriever       → recupera docs relevantes
   │       │
   │       └─→ formatear_docs → convierte docs en texto
   │
   └─→ RunnablePassthrough → pasa la pregunta tal cual
         │
         ▼
       prompt → combina context + question
         │
         ▼
        llm  → genera la respuesta
         │
         ▼
  StrOutputParser → extrae el texto limpio
```

In [9]:
# ── Función para formatear documentos recuperados ─────────────────
def formatear_docs(docs: list) -> str:
    """
    Convierte la lista de documentos recuperados por el retriever
    en un solo bloque de texto para el prompt.

    Cada documento se separa con doble salto de línea
    y se incluye su tema como encabezado.
    """
    return '\n\n'.join(
        f"[{doc.metadata['tema'].upper()}]\n{doc.page_content}"
        for doc in docs
    )

# ── Cadena RAG moderna con LCEL ───────────────────────────────────
rag_chain = (
    {
        'context' : retriever | formatear_docs,  # recupera y formatea el contexto
        'question': RunnablePassthrough()         # pasa la pregunta sin modificar
    }
    | prompt          # construye el prompt completo
    | llm             # envía a Mistral
    | StrOutputParser() # extrae el texto de la respuesta
)

print('✅ Cadena RAG lista')
print('   Flujo: retriever | formatear_docs | prompt | llm | StrOutputParser')

✅ Cadena RAG lista
   Flujo: retriever | formatear_docs | prompt | llm | StrOutputParser


## CELDA 9 — Probar con rag_chain.invoke

In [10]:
# rag_chain.invoke recibe directamente la pregunta como string
respuesta = rag_chain.invoke('¿Cuál fue el año con más ventas?')
print(respuesta)

El año con más ventas fue **2004**, con un total de **$4,724,162.60**.


In [11]:
respuesta = rag_chain.invoke('¿Qué línea de producto genera más ingresos?')
print(respuesta)

La línea de producto que genera más ingresos es **Classic Cars**, con un total de **$3,919,615.66**.


In [ ]:
respuesta = rag_chain.invoke('Dame un resumen ejecutivo de las ventas')
print(respuesta)

## CELDA 10 — Chat interactivo con rag_chain.invoke
> Escribe `salir` para terminar.

In [ ]:
import time

print('═══════════════════════════════════════════')
print('   AGENTE RAG — LangChain + FAISS + Mistral')
print('═══════════════════════════════════════════')
print('Escribe tu pregunta o "salir" para terminar.\n')

while True:
    pregunta = input('Tú: ').strip()

    if not pregunta:
        continue

    if pregunta.lower() in ['salir', 'exit', 'quit']:
        print('Agente: ¡Hasta luego!')
        break

    try:
        respuesta = rag_chain.invoke(pregunta)
        print(f'\nAgente: {respuesta}\n')
        print('─' * 45)
        time.sleep(2)  # evita el rate limit del plan gratuito
    except Exception as e:
        if '429' in str(e):
            print('⚠️  Rate limit — espera 10 segundos...\n')
            time.sleep(10)
        else:
            print(f'⚠️  Error: {e}\n')